# Proceso de validacion de clientes, sucursales y prestamos

Proposito del script: 
- Validar que las fechas de ingreso de los oficiales de credito sean anteriores a las fechas de otorgamiento de prestamos.  
- Modificar aquellos registros en oficiales de credito donde su fecha de ingreso sea posterior a fecha de otorgamiento del prestamo (se va a utilizar la fecha de otorgamiento para modificar la fecha de ingreso).
- Validar que estos cambios en las fechas de ingreso no afecte la relacion con la sucursal asignada. 
- Validar que las fechas de registros de los clientes sean anteriores a las fechas de otorgamientos de los prestamos.  
- Modificar aquellos registros en clientes donde su fecha de registro sea posterior a fecha de otorgamiento del prestamo (se va a utilizar la fecha de otorgamiento para modificar la fecha de registro).
- Validar que estos cambios en las fechas de registro no afecte la relacion con la sucursal asignada.

# Cargando los Archivos

In [1]:
import pandas as pd 
from conexiones_y_rutas import obtener_ruta_archivo
from datetime import date

df_clientes = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_clientes.parquet"))
df_clientes_tra = df_clientes.copy()

df_prestamos = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_prestamos.parquet"))
df_prestamos_tra = df_prestamos.copy()

df_oficial = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_oficiales_credito.parquet"))
df_oficial_tra = df_oficial.copy()

df_sucursal = pd.read_parquet(obtener_ruta_archivo("archivos_limpios","limpio_sucursales.parquet"))
df_sucursal_tra = df_sucursal.copy()

df_productos = pd.read_parquet(obtener_ruta_archivo("archivos_limpios","limpio_productos_crediticios.parquet"))
df_productos_tra = df_productos.copy()

# Proceso de Limpieza 

Como parto de datos que en su mayoria ya estan limpios, solo me voy a centrar en las columnas que se vean afectadas por los cambios en fecha_registro, fecha_nacimiento y fecha_ingreso.  

In [2]:
df_clientes_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   cliente_id                5000 non-null   int64         
 1   tipo_documento            5000 non-null   object        
 2   numero_documento          5000 non-null   object        
 3   nombres                   5000 non-null   object        
 4   apellido_paterno          5000 non-null   object        
 5   apellido_materno          5000 non-null   object        
 6   fecha_nacimiento          5000 non-null   datetime64[ns]
 7   edad                      5000 non-null   int32         
 8   genero                    5000 non-null   object        
 9   estado_civil              5000 non-null   object        
 10  nivel_educacion           5000 non-null   object        
 11  ocupacion                 5000 non-null   object        
 12  sector_economico    

In [3]:
df_prestamos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6495 entries, 0 to 6494
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   prestamo_id                   6495 non-null   int64         
 1   cliente_id                    6495 non-null   int64         
 2   sucursal_id                   6495 non-null   int64         
 3   producto_id                   6495 non-null   int64         
 4   oficial_id                    6495 non-null   int64         
 5   numero_contrato               6495 non-null   object        
 6   fecha_otorgamiento            6495 non-null   datetime64[ns]
 7   fecha_vencimiento             6495 non-null   datetime64[ns]
 8   monto_original                6495 non-null   float64       
 9   saldo_capital_vigente         6495 non-null   float64       
 10  tasa_interes_nominal_anual    6495 non-null   float64       
 11  tasa_interes_efectiva_anual   

In [4]:
df_oficial_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   oficial_id        60 non-null     int64         
 1   sucursal_id       60 non-null     int64         
 2   nombres           60 non-null     object        
 3   apellido_paterno  60 non-null     object        
 4   apellido_materno  60 non-null     object        
 5   genero            60 non-null     object        
 6   cargo             60 non-null     object        
 7   fecha_ingreso     60 non-null     datetime64[ns]
 8   estado            60 non-null     object        
dtypes: datetime64[ns](1), int64(2), object(6)
memory usage: 4.3+ KB


In [5]:
df_sucursal_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   sucursal_id      24 non-null     int64         
 1   codigo_sucursal  24 non-null     object        
 2   nombre_sucursal  24 non-null     object        
 3   tipo_sucursal    24 non-null     object        
 4   ciudad           24 non-null     object        
 5   departamento     24 non-null     object        
 6   region           24 non-null     object        
 7   zona             24 non-null     object        
 8   fecha_apertura   24 non-null     datetime64[ns]
 9   estado_sucursal  24 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 2.0+ KB


## oficial_credito

In [6]:
# LEFT JOIN, por oficial_id 
df_veri_fecha_ingreso = df_prestamos_tra[['prestamo_id','oficial_id','fecha_otorgamiento']].copy()
df_veri_fecha_ingreso_ofi = df_oficial_tra[['oficial_id','fecha_ingreso']].copy()
df_merge_veri_fecha_ingreso = df_veri_fecha_ingreso.merge(
    right= df_veri_fecha_ingreso_ofi,
    on="oficial_id",
    how='left'
)
# Verifica que la fecha_otorgamiento no sea mayor a la fecha_ingreso
# Resultados Esperados: Tabla Vacia 
df_revisar_fecha_ingreso = df_merge_veri_fecha_ingreso[
    (df_merge_veri_fecha_ingreso.fecha_otorgamiento 
        < df_merge_veri_fecha_ingreso.fecha_ingreso)
].copy()
df_revisar_fecha_ingreso

,prestamo_id,oficial_id,fecha_otorgamiento,fecha_ingreso
5,6,5,2020-08-07,2022-08-04
8,9,26,2020-02-20,2021-11-29
13,14,50,2022-01-02,2022-04-16
29,30,52,2022-12-30,2024-03-13
32,33,45,2020-06-05,2023-05-20
...,...,...,...,...
6465,6470,54,2022-05-17,2024-02-16
6478,6483,52,2022-02-08,2024-03-13
6483,6488,28,2020-11-26,2023-02-03
6484,6489,10,2020-09-14,2024-10-01


In [7]:
# Minima fecha de otorgamiento por cada oficial_id 
df_oc_fecha_otor_min = df_revisar_fecha_ingreso.groupby('oficial_id')['fecha_otorgamiento'].min()
# Nueva fecha de ingreso = fecha_otorgamiento_minima - 1 
df_oc_fecha_otor_min = (df_oc_fecha_otor_min - pd.DateOffset(days=1)).to_dict()
mascara_filtro_sucursales = df_oficial_tra.oficial_id.isin(df_oc_fecha_otor_min)
mascara_filtro_sucursales

0     False
1     False
2     False
3      True
4      True
5     False
6     False
7     False
8     False
9      True
10     True
11    False
12    False
13     True
14     True
15    False
16    False
17     True
18    False
19    False
20    False
21     True
22    False
23     True
24    False
25     True
26    False
27     True
28    False
29     True
30    False
31     True
32    False
33    False
34    False
35    False
36    False
37     True
38    False
39    False
40    False
41    False
42    False
43     True
44     True
45     True
46    False
47    False
48    False
49     True
50    False
51     True
52    False
53     True
54    False
55     True
56     True
57    False
58    False
59    False
Name: oficial_id, dtype: bool

In [8]:
# Reemplaza las fechas de registros anteriores  
df_oficial_tra.loc[mascara_filtro_sucursales,'fecha_ingreso'] = df_oficial_tra.loc[mascara_filtro_sucursales,'oficial_id'].map(df_oc_fecha_otor_min)
df_oficial_tra

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado
0,1,12,Alejandra,Herrera,Rivera,Femenino,Analista de Créditos,2017-12-20,Activo
1,2,24,Víctor,Rivera,Cusi,Masculino,Analista de Créditos,2015-06-29,Activo
2,3,8,Héctor,Vargas,López,Masculino,n/a,2010-09-20,Activo
3,4,7,Fernando,Laime,Vargas,Masculino,Analista Senior de Créditos,2020-01-01,Activo
4,5,8,Patricia,Ortiz,Ramírez,Femenino,n/a,2020-01-09,Activo
5,6,8,Fabiola,Rojas,Herrera,Femenino,Analista Senior de Créditos,2013-02-07,Activo
6,7,22,Miguel,Flores,Rivera,Masculino,Oficial de Créditos,2019-06-21,Activo
7,8,1,Isabel,Ortiz,Choque,Femenino,Oficial de Créditos,2012-07-27,Activo
8,9,10,Ángela,Castillo,Torres,Femenino,Analista de Créditos,2019-10-02,Activo
9,10,4,Isabel,Gómez,Condori,Femenino,Analista de Créditos,2020-01-01,Activo


### Verificando que no afecte la relacion con sucursal

In [9]:
# Columnas a utilizar
df_oficiales_revi = df_oficial_tra[['oficial_id','sucursal_id','fecha_ingreso']].copy()
df_sucursal_revi = df_sucursal_tra[['sucursal_id','fecha_apertura']].copy()
# LEFT JOIN
df_merge_fechas = df_oficiales_revi.merge(
    right = df_sucursal_revi,
    how='left',
    on='sucursal_id'
)
# Verifica si existen registros donde la fecha de ingreso sea anterior a la fecha de apertura 
# Resultados Esperados: Tabla Vacia
df_revisar_oficiales_cre = df_merge_fechas[df_merge_fechas.fecha_ingreso < df_merge_fechas.fecha_apertura].copy()
df_revisar_oficiales_cre

,oficial_id,sucursal_id,fecha_ingreso,fecha_apertura


In [10]:
# Fechas de ingreso futuras 
# Resultados Esperados: Tabla Vacia 
display(df_oficial_tra[df_oficial_tra.fecha_ingreso.dt.date >= date.today()])
# Fechas de otorgamiento futuras 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.fecha_otorgamiento.dt.date >= date.today()]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado


,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## cliente

### fecha_registro

In [11]:
# LEFT JOIN, cliente_id 
verificar_fecha_otor = df_prestamos_tra.merge(
    right= df_clientes_tra,
    on="cliente_id",
    how='left'
)
# Verifica que la fecha_otorgamiento no sea menor a la fecha_registro 
# Resultados Esperados: Tabla Vacia 
df_para_verificar = verificar_fecha_otor[
    (verificar_fecha_otor.fecha_otorgamiento 
        < verificar_fecha_otor.fecha_registro)
]
df_para_verificar[['prestamo_id','cliente_id','fecha_otorgamiento','fecha_registro']]

,prestamo_id,cliente_id,fecha_otorgamiento,fecha_registro
276,278,1362,2020-01-08,2032-07-20
1961,1965,1362,2024-01-17,2032-07-20


In [12]:
# fecha_otorgamiento minima para cada cliente 
df_fechas_minimas = df_para_verificar.groupby('cliente_id')['fecha_otorgamiento'].min()
# fecha_ingreso_nueva  = fecha_otorgamiento_minima - 1 dia
df_fechas_minimas = (df_fechas_minimas - pd.DateOffset(days=1)).to_dict()
mascara_filtro_clientes = df_clientes_tra.cliente_id.isin(df_fechas_minimas)
mascara_filtro_clientes

0       False
1       False
2       False
3       False
4       False
        ...  
4995    False
4996    False
4997    False
4998    False
4999    False
Name: cliente_id, Length: 5000, dtype: bool

In [13]:
# Reemplaza los valores que eran incorrectos 
df_clientes_tra.loc[mascara_filtro_clientes,'fecha_registro'] = df_clientes_tra.loc[mascara_filtro_clientes,'cliente_id'].map(df_fechas_minimas)
df_clientes_tra

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
0,1,DNI,60366909,Carmen,Ramírez,Flores,1977-09-16,48,Femenino,Casado,...,2072.19,888.76,39631.77,550,Regular,Agencia,122,2016-05-07,14,Activo
1,2,DNI,62729806,Cecilia,Cusi,Cusi,1964-03-13,62,Femenino,Casado,...,5663.68,3037.05,125691.99,528,Regular,Digital,105,2017-10-18,13,Activo
2,3,CE,641708053,Juan,Ortiz,Medina,1964-08-01,62,Masculino,Casado,...,4750.66,2302.13,226055.14,493,Regular,Agencia,90,2019-01-08,13,Activo
3,4,DNI,29912419,Carlos,Flores,Morales,1976-11-09,49,Masculino,Soltero,...,1832.75,763.53,104986.09,490,Regular,Telemarketing,196,2010-03-12,15,Activo
4,5,DNI,86518506,Sandra,González,Morales,1980-11-20,45,Femenino,Viudo,...,850.00,405.03,37044.25,392,Regular,Digital,162,2013-01-08,4,Activo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,DNI,91388532,Fernando,Pérez,Medina,1982-07-24,44,Masculino,Divorciado,...,11389.77,6588.40,484617.38,582,Regular,Agencia,63,2021-04-19,3,Activo
4996,4997,DNI,27375209,Sofía,Huanca,López,1972-06-28,54,Femenino,Soltero,...,1999.44,1392.12,16121.28,596,Regular,Agencia,159,2013-04-08,13,Bloqueado
4997,4998,DNI,79495936,Rodrigo,Laime,Mamani,1972-07-28,54,Masculino,Casado,...,7008.60,3344.03,371498.23,545,Premium,Telemarketing,87,2019-05-01,20,Activo
4998,4999,CE,925666317,Sandra,Sánchez,Apaza,1987-01-05,39,Femenino,Casado,...,1752.66,1063.05,87596.53,548,Regular,Agencia,68,2020-11-22,22,Activo


### antiguedad_cliente_meses

In [14]:
# Recalcula la antiguedad de los clientes en meses  
fecha_actual = date.today()

df_clientes_tra['antiguedad_cliente_meses'] = (
    (fecha_actual.year - df_clientes_tra.fecha_registro.dt.year) * 12
    + (fecha_actual.month - df_clientes_tra.fecha_registro.dt.month)
    - ( fecha_actual.day < df_clientes_tra.fecha_registro.dt.day)
)

df_clientes_tra[['cliente_id','fecha_registro','antiguedad_cliente_meses']]

,cliente_id,fecha_registro,antiguedad_cliente_meses
0,1,2016-05-07,122
1,2,2017-10-18,105
2,3,2019-01-08,90
3,4,2010-03-12,196
4,5,2013-01-08,162
...,...,...,...
4995,4996,2021-04-19,63
4996,4997,2013-04-08,159
4997,4998,2019-05-01,87
4998,4999,2020-11-22,68


### fecha_nacimiento

In [15]:
# Verifica que la fecha de registro sea cuando el cliente tiene 18 años o mas 
prueba_fechas = df_clientes_tra[["cliente_id","fecha_nacimiento","fecha_registro"]].copy()
prueba_fechas["edad_registro"] = (
    prueba_fechas.fecha_registro.dt.year
    - prueba_fechas.fecha_nacimiento.dt.year
    - ( 
        (prueba_fechas.fecha_registro.dt.month < prueba_fechas.fecha_nacimiento.dt.month)
        |
        (
            (prueba_fechas.fecha_registro.dt.month == prueba_fechas.fecha_nacimiento.dt.month)
            &
            (prueba_fechas.fecha_registro.dt.day < prueba_fechas.fecha_nacimiento.dt.day)
        )
    )
)

# Selecciona los registros con fechas erroneas 
prueba_fechas = prueba_fechas[prueba_fechas.edad_registro <18]
prueba_fechas

,cliente_id,fecha_nacimiento,fecha_registro,edad_registro
1361,1362,2014-07-20,2020-01-07,5


In [16]:
# Filtra las fechas incorrectas, ahora la nueva fecha de nacimiento, para estos datos que no cumplen va a ser la fecha de registro - 18 años - 1 dia
df_limpieza_fechas = prueba_fechas.copy()
df_limpieza_fechas["fecha_nacimiento_correcta"] = df_limpieza_fechas.fecha_registro - pd.DateOffset(years=18,days=1)
df_limpieza_fechas

,cliente_id,fecha_nacimiento,fecha_registro,edad_registro,fecha_nacimiento_correcta
1361,1362,2014-07-20,2020-01-07,5,2002-01-06


In [17]:
# Crea un diccionario llave: cliente_id, valor: fecha_nacimiento
df_fechas_registro = (
    df_limpieza_fechas
    .set_index('cliente_id')['fecha_nacimiento_correcta']
    .to_dict()
)
mascara_filtro_fecha_registro = df_clientes_tra.cliente_id.isin(df_fechas_registro)
mascara_filtro_fecha_registro

0       False
1       False
2       False
3       False
4       False
        ...  
4995    False
4996    False
4997    False
4998    False
4999    False
Name: cliente_id, Length: 5000, dtype: bool

In [18]:
# Reemplaza los valores incorrectos
df_clientes_tra.loc[mascara_filtro_fecha_registro,'fecha_nacimiento'] = df_clientes_tra.loc[mascara_filtro_fecha_registro,'cliente_id'].map(df_fechas_registro)
df_clientes_tra

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
0,1,DNI,60366909,Carmen,Ramírez,Flores,1977-09-16,48,Femenino,Casado,...,2072.19,888.76,39631.77,550,Regular,Agencia,122,2016-05-07,14,Activo
1,2,DNI,62729806,Cecilia,Cusi,Cusi,1964-03-13,62,Femenino,Casado,...,5663.68,3037.05,125691.99,528,Regular,Digital,105,2017-10-18,13,Activo
2,3,CE,641708053,Juan,Ortiz,Medina,1964-08-01,62,Masculino,Casado,...,4750.66,2302.13,226055.14,493,Regular,Agencia,90,2019-01-08,13,Activo
3,4,DNI,29912419,Carlos,Flores,Morales,1976-11-09,49,Masculino,Soltero,...,1832.75,763.53,104986.09,490,Regular,Telemarketing,196,2010-03-12,15,Activo
4,5,DNI,86518506,Sandra,González,Morales,1980-11-20,45,Femenino,Viudo,...,850.00,405.03,37044.25,392,Regular,Digital,162,2013-01-08,4,Activo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,DNI,91388532,Fernando,Pérez,Medina,1982-07-24,44,Masculino,Divorciado,...,11389.77,6588.40,484617.38,582,Regular,Agencia,63,2021-04-19,3,Activo
4996,4997,DNI,27375209,Sofía,Huanca,López,1972-06-28,54,Femenino,Soltero,...,1999.44,1392.12,16121.28,596,Regular,Agencia,159,2013-04-08,13,Bloqueado
4997,4998,DNI,79495936,Rodrigo,Laime,Mamani,1972-07-28,54,Masculino,Casado,...,7008.60,3344.03,371498.23,545,Premium,Telemarketing,87,2019-05-01,20,Activo
4998,4999,CE,925666317,Sandra,Sánchez,Apaza,1987-01-05,39,Femenino,Casado,...,1752.66,1063.05,87596.53,548,Regular,Agencia,68,2020-11-22,22,Activo


### edad

In [19]:
# Recalcula la edad, ya que es un dato que esta desactualizado 
# Realmente no se tienen que guardar edad, salvo casos especificos como fecha de corte
fecha_actual = date.today()
df_clientes_tra['edad'] = (
    fecha_actual.year - df_clientes_tra.fecha_nacimiento.dt.year
    - (
        (fecha_actual.month < df_clientes_tra.fecha_nacimiento.dt.month)
        |
        (
            (fecha_actual.month == df_clientes_tra.fecha_nacimiento.dt.month)
            &
            (fecha_actual.day < df_clientes_tra.fecha_nacimiento.dt.day)
        )
    )
)
df_clientes_tra

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
0,1,DNI,60366909,Carmen,Ramírez,Flores,1977-09-16,48,Femenino,Casado,...,2072.19,888.76,39631.77,550,Regular,Agencia,122,2016-05-07,14,Activo
1,2,DNI,62729806,Cecilia,Cusi,Cusi,1964-03-13,62,Femenino,Casado,...,5663.68,3037.05,125691.99,528,Regular,Digital,105,2017-10-18,13,Activo
2,3,CE,641708053,Juan,Ortiz,Medina,1964-08-01,62,Masculino,Casado,...,4750.66,2302.13,226055.14,493,Regular,Agencia,90,2019-01-08,13,Activo
3,4,DNI,29912419,Carlos,Flores,Morales,1976-11-09,49,Masculino,Soltero,...,1832.75,763.53,104986.09,490,Regular,Telemarketing,196,2010-03-12,15,Activo
4,5,DNI,86518506,Sandra,González,Morales,1980-11-20,45,Femenino,Viudo,...,850.00,405.03,37044.25,392,Regular,Digital,162,2013-01-08,4,Activo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,DNI,91388532,Fernando,Pérez,Medina,1982-07-24,44,Masculino,Divorciado,...,11389.77,6588.40,484617.38,582,Regular,Agencia,63,2021-04-19,3,Activo
4996,4997,DNI,27375209,Sofía,Huanca,López,1972-06-28,54,Femenino,Soltero,...,1999.44,1392.12,16121.28,596,Regular,Agencia,159,2013-04-08,13,Bloqueado
4997,4998,DNI,79495936,Rodrigo,Laime,Mamani,1972-07-28,54,Masculino,Casado,...,7008.60,3344.03,371498.23,545,Premium,Telemarketing,87,2019-05-01,20,Activo
4998,4999,CE,925666317,Sandra,Sánchez,Apaza,1987-01-05,39,Femenino,Casado,...,1752.66,1063.05,87596.53,548,Regular,Agencia,68,2020-11-22,22,Activo


### Verificando que no afecte la relacion con sucursal

In [20]:
# Selecciona las columnas a utilizar
df_clientes_veri = df_clientes_tra[['cliente_id','sucursal_id','fecha_registro']].copy()
df_sucursales_veri = df_sucursal_tra[['sucursal_id','fecha_apertura']]
# LEFT JOIN, sucursal_id 
df_merge_veri_sucursal = df_clientes_veri.merge(
    right=df_sucursales_veri,
    on="sucursal_id",
    how='left')

# Verifica si existan registros donde la fecha de registro sea anterior a la fecha de nacimiento
# Resultados Esperados: Tabla Vacia 
df_merge_veri_sucursal[df_merge_veri_sucursal.fecha_registro < df_merge_veri_sucursal.fecha_apertura]

,cliente_id,sucursal_id,fecha_registro,fecha_apertura


In [21]:
# Fechas de nacimiento futuras 
# Resultados Esperados: Tabla Vacia 
display(df_clientes_tra[df_clientes_tra.fecha_nacimiento.dt.date >= date.today()])
# Fechas de registro futuras 
# Resultados Esperados: Tabla Vacia 
df_clientes_tra[df_clientes_tra.fecha_registro.dt.date >= date.today()]

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente


,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente


# Carga los Nuevos Registros y Sobrescribe el archivo original 

In [22]:
df_prestamos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6495 entries, 0 to 6494
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   prestamo_id                   6495 non-null   int64         
 1   cliente_id                    6495 non-null   int64         
 2   sucursal_id                   6495 non-null   int64         
 3   producto_id                   6495 non-null   int64         
 4   oficial_id                    6495 non-null   int64         
 5   numero_contrato               6495 non-null   object        
 6   fecha_otorgamiento            6495 non-null   datetime64[ns]
 7   fecha_vencimiento             6495 non-null   datetime64[ns]
 8   monto_original                6495 non-null   float64       
 9   saldo_capital_vigente         6495 non-null   float64       
 10  tasa_interes_nominal_anual    6495 non-null   float64       
 11  tasa_interes_efectiva_anual   

In [23]:
df_oficial_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   oficial_id        60 non-null     int64         
 1   sucursal_id       60 non-null     int64         
 2   nombres           60 non-null     object        
 3   apellido_paterno  60 non-null     object        
 4   apellido_materno  60 non-null     object        
 5   genero            60 non-null     object        
 6   cargo             60 non-null     object        
 7   fecha_ingreso     60 non-null     datetime64[ns]
 8   estado            60 non-null     object        
dtypes: datetime64[ns](1), int64(2), object(6)
memory usage: 4.3+ KB


In [24]:
df_clientes_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   cliente_id                5000 non-null   int64         
 1   tipo_documento            5000 non-null   object        
 2   numero_documento          5000 non-null   object        
 3   nombres                   5000 non-null   object        
 4   apellido_paterno          5000 non-null   object        
 5   apellido_materno          5000 non-null   object        
 6   fecha_nacimiento          5000 non-null   datetime64[ns]
 7   edad                      5000 non-null   int32         
 8   genero                    5000 non-null   object        
 9   estado_civil              5000 non-null   object        
 10  nivel_educacion           5000 non-null   object        
 11  ocupacion                 5000 non-null   object        
 12  sector_economico    

In [25]:
df_clientes_tra.to_parquet(
    obtener_ruta_archivo("archivos_limpios","limpio_clientes.parquet"),
    index=False
)

In [26]:
df_oficial_tra.to_parquet(
    obtener_ruta_archivo("archivos_limpios","limpio_oficiales_credito.parquet"),
    index=False
)